# Progress Report 2 — Fraud Detection Modeling

**Student:** Vamsi Doddapaneni  
**StudentID:** 3121374  
**Course:** PSYC 699 – Community Data Labs

This report focuses on the **technical progress made after Progress Report 1**. While the first report introduced the PaySim dataset and baseline models, this report documents new developments including temporal validation, feature engineering, and robustness analysis.


## 1. Brief Recap of Progress Report 1

Progress Report 1 introduced the PaySim financial transaction dataset and explored the severe class imbalance present in fraud detection problems. Initial baseline models were trained using Random Forest and XGBoost classifiers. Evaluation using precision‑recall metrics demonstrated that XGBoost produced stronger performance than Random Forest.

The main limitation identified in Report 1 was that models were evaluated using standard splits, which may not reflect realistic fraud detection conditions. Additionally, only raw transaction features were used. The work in this report addresses these limitations.

## 2. Advancement 1 — Temporal Train/Test Splitting

A key improvement since the first report is the transition from random data splits to **time‑aware model evaluation**. Financial transactions occur sequentially, so training a model on future data while testing on earlier data introduces information leakage.

The dataset includes a `step` variable representing transaction time. The dataset was sorted by this variable and split chronologically.


In [1]:
import pandas as pd
df = pd.read_csv('transaction_data.csv')
df = df.sort_values('step').reset_index(drop=True)

split_index = int(len(df)*0.80)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

print('Train size:', len(train_df))
print('Test size:', len(test_df))


Train size: 5090096
Test size: 1272524


### Temporal Sensitivity Testing

Beyond a single split, additional experiments were conducted to measure **model robustness under temporal distribution shift**.

Multiple training windows were evaluated:

- Train on first 30% → Test on next 10%
- Train on first 60% → Test on next 10%
- Train on first 90% → Test on next 10%

This approach evaluates whether fraud patterns learned earlier in time remain predictive for later transactions.

In [2]:
df['delta_org'] = df['newbalanceOrig'] - df['oldbalanceOrg']
df['delta_dest'] = df['newbalanceDest'] - df['oldbalanceDest']
df['amount_balance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)
df[['amount','delta_org','delta_dest','amount_balance_ratio']].head()


,amount,delta_org,delta_dest,amount_balance_ratio
0,9839.64,-9839.64,0.0,0.057834
1,1864.28,-1864.28,0.0,0.087731
2,181.00,-181.00,0.0,0.994505
3,181.00,-181.00,-21182.0,0.994505
4,11668.14,-11668.14,0.0,0.280788


## 3. Advancement 2 — Feature Splicing and Behavioral Features

A major improvement since Report 1 is the introduction of **derived behavioral features** rather than relying only on raw transaction variables.

Fraud detection often depends on identifying **unusual behavior relative to account history**, rather than simply the raw transaction amount.


### Account Prefix Parsing

Account identifiers in PaySim contain encoded information about account type.

`C12345` → customer account  
`M67890` → merchant account

Extracting this prefix allows the model to learn context about the entities involved in each transaction.

In [3]:
df['orig_prefix'] = df['nameOrig'].astype(str).str[0]
df['dest_prefix'] = df['nameDest'].astype(str).str[0]
df[['nameOrig','nameDest','orig_prefix','dest_prefix']].head()


,nameOrig,nameDest,orig_prefix,dest_prefix
0,C1231006815,M1979787155,C,M
1,C1666544295,M2044282225,C,M
2,C1305486145,C553264065,C,C
3,C840083671,C38997010,C,C
4,C2048537720,M1230701703,C,M


### Behavioral / Panel Features

The project also introduces **panel‑style behavioral features** capturing transaction history for each account.

Examples include:

- sender transaction count so far
- cumulative transfer amount from an account
- transaction frequency patterns

These features help detect accounts that suddenly behave abnormally.

## 4. Advancement 3 — Panel + Temporal XGBoost Model

The latest model combines engineered behavioral features with a temporally‑aware XGBoost classifier. The pipeline now includes:

- time‑ordered training/testing
- behavioral feature engineering
- handling extreme class imbalance
- evaluation using AUPRC


## 5. New Research Question: Simulator Overfitting

An important concern raised during project discussions is whether the model is learning real fraud behavior or simply learning the rules embedded in the PaySim simulator.

Since PaySim generates fraud using deterministic logic, a model may achieve extremely high performance by learning those rules instead of learning generalizable fraud indicators.

## 6. Planned Experiment — Custom Synthetic Dataset

To address this concern, the next phase of the project will include generating a **custom synthetic fraud dataset** with the same schema but different fraud patterns.

Example variations:

- multiple medium‑size transfers
- partial account withdrawals
- fraud interacting with several destination accounts
- bursts of suspicious activity


### Cross‑Dataset Evaluation Plan

| Train Data | Test Data | Goal |
|-----------|-----------|------|
| PaySim | PaySim | baseline |
| Custom | Custom | validate generator |
| PaySim | Custom | test generalization |

This experiment will determine whether the model captures general fraud behavior or PaySim‑specific simulation patterns.

## 7. Conclusion

Since Progress Report 1, the project has progressed through temporal model evaluation and behavioral feature engineering. The updated model demonstrates strong fraud detection performance. However, testing with an independent synthetic dataset will be critical for confirming the generalizability of the approach.

## Appendix: Code Repository

The code for this project is available in the project repository including:

- data preprocessing scripts
- Random Forest baseline
- XGBoost model implementation
- temporal sensitivity experiments